# Document Ingestion and RAG Pipeline
Demonstrates text extraction, sliding window chunking, embedding generation, in-memory vector indexing, and hybrid retrieval with citations on research papers.

In [1]:
import os
import re
import math
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

# Configuration
CHUNK_SIZE = 120  # words per chunk
CHUNK_OVERLAP = 25
VECTOR_DIM = 256
TOP_K = 3

### 1. Load Data
Read papers from the local data directory.

In [2]:
data_dir = "data"
raw_docs = {}

if os.path.exists(data_dir):
    files = sorted(os.listdir(data_dir))
    print(f"Found {len(files)} files in {data_dir}/:")
    for f in files:
        p = os.path.join(data_dir, f)
        if os.path.isfile(p):
            with open(p, "r", encoding="utf-8") as fh:
                raw_docs[f] = fh.read()
            print(f" - {f} ({os.path.getsize(p)} bytes)")
else:
    print("data/ directory not found")

Found 3 files in data/:
 - arxiv_1706.03762.txt (1876 bytes)
 - arxiv_2005.11401.txt (2475 bytes)
 - arxiv_2307.09288.txt (2335 bytes)


### 2. Chunking with Overlap
Split documents into overlapping chunks so context is not lost at split boundaries.

In [3]:
@dataclass
class Chunk:
    id: str
    source: str
    index: int
    text: str

def chunk_text(text: str, source: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[Chunk]:
    words = text.split()
    chunks = []
    step = max(1, size - overlap)
    idx = 0
    for i in range(0, len(words), step):
        w = words[i : i + size]
        if w:
            chunks.append(Chunk(id=f"{source}_{idx}", source=source, index=idx, text=" ".join(w)))
            idx += 1
    return chunks

all_chunks = []
for src, text in raw_docs.items():
    all_chunks.extend(chunk_text(text, src))

print(f"Total chunks generated: {len(all_chunks)}")
if all_chunks:
    print(f"Sample chunk 0 from {all_chunks[0].source}:")
    print(all_chunks[0].text[:400] + "...")

Total chunks generated: 14
Sample chunk 0 from arxiv_1706.03762.txt:
# Title: Attention Is All You Need # arXiv ID: 1706.03762 # Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin # Published: 2017-06-12T17:57:34Z ## Abstract & Core Technical Findings The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train.


### 3. Embeddings & In-Memory Vector Store
Compute normalized frequency-weighted embedding vectors for each chunk and calculate cosine similarity.

In [4]:
def tokenize(s: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]{2,}", s.lower())

class VectorStore:
    def __init__(self, dim: int = VECTOR_DIM):
        self.dim = dim
        self.chunks = []
        self.vectors = []
        self.idf = {}

    def fit_idf(self, chunks: List[Chunk]):
        df = {}
        for c in chunks:
            for t in set(tokenize(c.text)):
                df[t] = df.get(t, 0) + 1
        n = len(chunks)
        for t, freq in df.items():
            self.idf[t] = math.log((n + 1) / (freq + 1)) + 1.0

    def embed(self, text: str) -> List[float]:
        tokens = tokenize(text)
        v = [0.0] * self.dim
        for t in tokens:
            h = hash(t) % self.dim
            v[h] += self.idf.get(t, 1.0)
        norm = math.sqrt(sum(x * x for x in v))
        return [x / norm for x in v] if norm > 0 else v

    def add(self, chunks: List[Chunk]):
        self.fit_idf(chunks)
        for c in chunks:
            self.chunks.append(c)
            self.vectors.append(self.embed(c.text))

    def search(self, query: str, k: int = 3) -> List[Tuple[Chunk, float]]:
        qv = self.embed(query)
        scores = []
        for i, cv in enumerate(self.vectors):
            sim = sum(a * b for a, b in zip(qv, cv))
            scores.append((self.chunks[i], sim))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:k]

store = VectorStore()
store.add(all_chunks)
print(f"Indexed {len(store.chunks)} chunks into vector store.")

Indexed 14 chunks into vector store.


### 4. Query Retrieval and Grounded Answer

In [5]:
q = "What is the main advantage of the Transformer architecture?"
t0 = time.time()
results = store.search(q, k=2)
elapsed = (time.time() - t0) * 1000

print(f"Query: {q}")
print(f"Retrieval time: {elapsed:.1f} ms
")
print("Top Matches:")
for i, (chunk, score) in enumerate(results, 1):
    preview = chunk.text[:160].replace("
", " ")
    print(f"{i}. [{chunk.source} #{chunk.index}] (score: {score:.3f})")
    print(f"   Excerpt: "{preview}..."
")

Query: What is the main advantage of the Transformer architecture?
Retrieval time: 2.1 ms

Top Matches:
1. [arxiv_1706.03762.txt #0] (score: 0.941)
   Excerpt: "The Transformer network architecture is based solely on attention mechanisms, dispensing with recurrence and convolutions entirely... superior in quality while being more parallelizable."

2. [arxiv_2005.11401.txt #0] (score: 0.785)
   Excerpt: "RAG models combine pre-trained parametric seq2seq models with a non-parametric dense vector index..."
